# 04 -- Sentiment analysis

Classifies comments as positive, neutral or negative with an Italian BERT model ([neuraly/bert-base-italian-cased-sentiment](https://huggingface.co/neuraly/bert-base-italian-cased-sentiment)) and saves one label per comment to `analytics/output/litigi_comment_sentiment.parquet`, which `05_posting_habits.ipynb` reads.

A GPU is used automatically when available. On a CPU the model classifies about 10 comments per second, so classifying all of the ~480,000 comments takes more than 10 hours: by default the notebook classifies a random sample of `SAMPLE_SIZE` comments.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import pipeline

from subreddit_lens import load_comments, preprocess
from subreddit_lens.constants import DEFAULT_EXCLUDED_AUTHORS, REMOVED_BODIES

from subreddit_lens import load_config

# Locate the analytics directory, which holds subreddit-lens.toml, so paths
# work regardless of the working directory.
ANALYTICS_DIR = next(
    p
    for p in [Path.cwd(), Path.cwd() / "analytics", *Path.cwd().parents]
    if (p / "subreddit-lens.toml").exists()
)
config = load_config(ANALYTICS_DIR / "subreddit-lens.toml")
DATA_DIR = config.data_dir
OUTPUT_DIR = config.output_dir
OUTPUT_DIR.mkdir(exist_ok=True)

# Number of comments to classify, sampled at random. None classifies all of
# them (hours on a CPU, see above).
SAMPLE_SIZE = 2_000
# Comments per model call. Large batches pay off on a GPU; on a CPU, padding
# every comment to the longest one in its batch makes them slower.
BATCH_SIZE = 64 if torch.cuda.is_available() else 8

In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="neuraly/bert-base-italian-cased-sentiment",
    # Comments longer than the model's 512 tokens are truncated.
    truncation=True,
    max_length=512,
    device=0 if torch.cuda.is_available() else -1,
)
classifier(["Grazie mille, sei stato gentilissimo!", "Sei proprio un idiota."])

## Load the comments

In [ ]:
litigi = load_comments(DATA_DIR / "litigi_comments.parquet")
# Deleted and removed comments have no text to classify, and neither have
# comments that only quote other comments.
litigi = litigi[~litigi["body"].isin(REMOVED_BODIES)]
litigi["body_preprocessed"] = litigi["body"].map(preprocess)
litigi = litigi[litigi["body_preprocessed"] != ""]
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(litigi):
    litigi = litigi.sample(SAMPLE_SIZE, random_state=0)
litigi = litigi.reset_index(drop=True)
print(f"{len(litigi):,} comments to classify")
litigi[["author", "body", "body_preprocessed"]]

## Classify

In [ ]:
texts = litigi["body_preprocessed"].tolist()
results = []
for start in tqdm(range(0, len(texts), BATCH_SIZE), unit="batch"):
    results += classifier(texts[start : start + BATCH_SIZE], batch_size=BATCH_SIZE)

litigi["sentiment"] = [r["label"] for r in results]
litigi["sentiment_score"] = [r["score"] for r in results]
litigi["num_words"] = litigi["body_preprocessed"].str.split().str.len()
litigi["sentiment"].value_counts()

## Results

In [ ]:
# Number of comments per author and sentiment label.
df_pivot = pd.pivot_table(
    litigi, index="author", columns="sentiment", aggfunc="size", fill_value=0
).sort_values("positive", ascending=False)
df_pivot.to_csv(OUTPUT_DIR / "user_sentiment_litigi.csv")
df_pivot

In [ ]:
# Comments of the most active author among the classified ones.
authors = litigi.loc[~litigi["author"].isin(DEFAULT_EXCLUDED_AUTHORS), "author"]
author = authors.value_counts().index[0]
litigi.loc[litigi["author"] == author, ["author", "body", "sentiment", "num_words"]]

In [ ]:
# One label per comment, read by 05_posting_habits.ipynb.
litigi[["id", "sentiment", "sentiment_score"]].to_parquet(
    OUTPUT_DIR / "litigi_comment_sentiment.parquet", index=False
)